# 30 – msgspec High-Performance Serialization

## Learning objectives

- Define immutable and wire-friendly structs with `msgspec.Struct`.
- Encode and decode JSON and MessagePack payloads.
- Interpret a simple benchmark versus Pydantic and the stdlib `json` module.

## About this topic

msgspec focuses on fast, schema-driven serialization. Structs map closely to C structs: slots, optional immutability (`frozen=True`), and strict decoding (`forbid_unknown_fields=True`). It complements Pydantic: use Pydantic at API boundaries where rich validation matters, and msgspec inside hot loops or service meshes where throughput dominates.

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/msgspec_structs/basic_structs.py"
spec = importlib.util.spec_from_file_location("ms_basic", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

In [ ]:
import msgspec

class Event(msgspec.Struct, frozen=True, forbid_unknown_fields=True):
    name: str
    score: float

payload = Event(name="ingest", score=1.25)
js = msgspec.json.encode(payload)
print("json bytes", js)
print("roundtrip", msgspec.json.decode(js, type=Event))
mp = msgspec.msgpack.encode(payload)
print("msgpack len", len(mp), "decoded", msgspec.msgpack.decode(mp, type=Event))

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/msgspec_structs/serialization_benchmark.py"
spec = importlib.util.spec_from_file_location("ms_bench", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

## Exercises and next steps

1. Add an optional `meta: dict[str, str]` field to `Event` and measure encode/decode time delta in the benchmark script.
2. Read `examples/library/litestar_patterns/msgspec_integration.py` and note how Litestar plugs msgspec into handlers.
3. List one case where you would still choose Pydantic over msgspec for a public HTTP API.